# docx-extract OCR server (DeepSeek-OCR-2) trên Google Colab (T4)

Notebook này chạy server FastAPI OCR cho MathType OLE (WMF/EMF) -> LaTeX,
dùng bởi `src/docx_extract/mathocr.py` qua biến môi trường `DOCX_EXTRACT_OCR_ENDPOINT`.

**Trước khi chạy:**
1. Đảm bảo `Runtime > Change runtime type > T4 GPU` đã được chọn.
2. Copy thư mục `colab_ocr_server/` (chứa `ocr_server.py`, `ocr_logic.py`) vào Google Drive,
   ví dụ tại `My Drive/docx-extract/colab_ocr_server/`.
3. Có sẵn ngrok authtoken (free tại https://ngrok.com) để dán vào Cell 3.

## Cell 1: Mount Google Drive và trỏ vào thư mục code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sửa đường dẫn này nếu bạn để thư mục ở vị trí khác trong Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/docx-extract/colab_ocr_server"

%cd "$DRIVE_PROJECT_DIR"
!ls -la

## Cell 2: Cài đặt dependencies (chạy 1 lần / phiên Colab)

In [ ]:
!apt-get -qq update && apt-get -qq install -y libreoffice-core libreoffice-impress
# Khong pin torch/torchvision: Colab da co san ban tuong thich voi GPU driver/CUDA cua may.
# Pin them torch se gay xung dot voi torchvision co san (vd: torchvision requires torch==2.11.0).
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok nest_asyncio \
    transformers==4.46.3 tokenizers==0.20.3 einops addict easydict

## Cell 3: Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## Cell 4: Chạy server + expose ra internet qua ngrok

Thay `NGROK_AUTHTOKEN` bằng token thật của bạn (lấy tại https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
import nest_asyncio, uvicorn, threading, time
from pyngrok import ngrok

NGROK_AUTHTOKEN = "DAN_TOKEN_NGROK_CUA_BAN_VAO_DAY"

nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Đóng tunnel cũ nếu chạy lại cell này nhiều lần trong cùng phiên
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(8000)
print("OCR endpoint:", f"{public_url}/ocr")
print("Health check:", f"{public_url}/health")

server_thread = threading.Thread(
    target=lambda: uvicorn.run("ocr_server:app", host="0.0.0.0", port=8000, log_level="info"),
    daemon=True,
)
server_thread.start()

time.sleep(3)  # chờ server khởi động trước khi gọi thử ở cell tiếp theo

## Cell 5: Kiểm tra nhanh /health

In [ ]:
import requests
r = requests.get(f"{public_url}/health", timeout=10)
print(r.status_code, r.text)

## Cell 6: Test OCR thử với 1 ảnh công thức

Upload 1 file ảnh công thức mẫu (PNG/WMF/EMF) lên Colab (kéo-thả vào panel Files bên trái,
hoặc để trong Drive rồi sửa đường dẫn `SAMPLE_FILE` dưới đây).

In [ ]:
SAMPLE_FILE = "sample_formula.png"  # sửa thành đường dẫn file ảnh công thức thật của bạn

with open(SAMPLE_FILE, "rb") as f:
    resp = requests.post(f"{public_url}/ocr", files={"file": (SAMPLE_FILE, f.read())}, timeout=60)
print(resp.status_code, resp.json())

## Cell 7: Benchmark thông lượng thật (đo số giây/công thức)

Chạy lặp lại N lần với ảnh mẫu để có số liệu thật cho workload của bạn, thay vì ước tính lý thuyết.

In [ ]:
import time

N = 20
with open(SAMPLE_FILE, "rb") as f:
    payload = f.read()

t0 = time.time()
for _ in range(N):
    requests.post(f"{public_url}/ocr", files={"file": (SAMPLE_FILE, payload)}, timeout=60)
elapsed = time.time() - t0

print(f"{elapsed / N:.2f} giay/cong thuc  ->  ~{N / elapsed * 60:.0f} cong thuc/phut")

## Cell 8: Lấy URL endpoint để dùng bên máy chạy docx-extract

Copy dòng lệnh in ra và chạy trên máy Windows của bạn (PowerShell) trước khi chạy `docx-extract`.

In [ ]:
print(f'$env:DOCX_EXTRACT_OCR_ENDPOINT = "{public_url}/ocr"')

---
### Ghi chú
- Phiên Colab free bị ngắt sau ~90 phút không tương tác, tối đa ~12 giờ. Giữ tab active nếu xử lý lô lớn.
- URL ngrok đổi mỗi lần chạy lại Cell 4 (free plan không có domain cố định) — cập nhật lại `DOCX_EXTRACT_OCR_ENDPOINT` mỗi phiên mới.
- Xem `README.md` trong cùng thư mục để biết chi tiết tính toán VRAM/thông lượng.